# Cleaning
<div style="height:1px;background:#C62828;"></div>


In [10]:
import pandas as pd
import sqlite3

db_path = "../data/raw/names.sqlite"
conn = sqlite3.connect(db_path)


In [11]:
def styled_table(df, caption, color="#C62828"):
    """Zeigt ein DataFrame als formatierte Tabelle mit farbiger Caption an."""
    return df.style.set_caption(caption).set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("background-color", color),
                ("color", "white"),
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("padding", "8px")
            ]
        }
    ])


In [12]:
df_national = pd.read_sql_query("SELECT * FROM NationalNames;", conn)
df_state = pd.read_sql_query("SELECT * FROM StateNames;", conn)


## NationalNames

In [13]:
missing_national = df_national.isnull().sum().to_frame(name="Missing Values")
display(styled_table(missing_national, "Missing Values - NationalNames"))


,Missing Values
Id,0
Name,0
Year,0
Gender,0
Count,0


In [14]:
# invalid year value check
query = """
SELECT * FROM NationalNames
WHERE Year < 1880 OR Year > 2014;
"""
df_invalid_years = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_years, "Invalid Year Values - NationalNames"))


,Id,Name,Year,Gender,Count


In [15]:
# invalid gender values
query = """
SELECT * FROM NationalNames
WHERE Gender NOT IN ('F', 'M');
"""
df_invalid_gender = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_gender, "Invalid Gender Values - NationalNames"))


,Id,Name,Year,Gender,Count


In [16]:
# invalid birth count
query = """
SELECT * FROM NationalNames
WHERE Count <= 0;
"""
df_invalid_count = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_count, "Invalid Birth Counts - NationalNames"))


,Id,Name,Year,Gender,Count


In [17]:
# empty names
query = """
SELECT * FROM NationalNames
WHERE TRIM(Name) = '';
"""
df_empty_names = pd.read_sql_query(query, conn)
display(styled_table(df_empty_names, "Empty Names - NationalNames"))


,Id,Name,Year,Gender,Count


In [18]:
# duplicate records
query = """
SELECT Name, Year, Gender, COUNT(*) AS DuplicateCount
FROM NationalNames
GROUP BY Name, Year, Gender
HAVING COUNT(*) > 1;
"""
df_duplicates = pd.read_sql_query(query, conn)
display(styled_table(df_duplicates, "Duplicate Records - NationalNames"))


,Name,Year,Gender,DuplicateCount


## StateNames

In [19]:
missing_state = df_state.isnull().sum().to_frame(name="Missing Values")
display(styled_table(missing_state, "Missing Values - StateNames", "#2E7D32"))


,Missing Values
Id,0
Name,0
Year,0
Gender,0
State,0
Count,0


In [20]:
# invalid year value check
query = """
SELECT * FROM StateNames
WHERE Year < 1880 OR Year > 2014;
"""
df_invalid_years_state = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_years_state, "Invalid Year Values - StateNames", "#2E7D32"))


,Id,Name,Year,Gender,State,Count


In [21]:
# invalid gender values
query = """
SELECT * FROM StateNames
WHERE Gender NOT IN ('F', 'M');
"""
df_invalid_gender_state = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_gender_state, "Invalid Gender Values - StateNames", "#2E7D32"))


,Id,Name,Year,Gender,State,Count


In [22]:
# invalid birth count
query = """
SELECT * FROM StateNames
WHERE Count <= 0;
"""
df_invalid_count_state = pd.read_sql_query(query, conn)
display(styled_table(df_invalid_count_state, "Invalid Birth Counts - StateNames", "#2E7D32"))


,Id,Name,Year,Gender,State,Count


In [23]:
# empty names
query = """
SELECT * FROM StateNames
WHERE TRIM(Name) = '';
"""
df_empty_names_state = pd.read_sql_query(query, conn)
display(styled_table(df_empty_names_state, "Empty Names - StateNames", "#2E7D32"))


,Id,Name,Year,Gender,State,Count


In [24]:
# duplicate records
query = """
SELECT Name, Year, Gender, State, COUNT(*) AS DuplicateCount
FROM StateNames
GROUP BY Name, Year, Gender, State
HAVING COUNT(*) > 1;
"""
df_duplicates_state = pd.read_sql_query(query, conn)
display(styled_table(df_duplicates_state, "Duplicate Records - StateNames", "#2E7D32"))


,Name,Year,Gender,State,DuplicateCount


In [25]:
# invalid state codes (expect 50 states + DC = 51)
query = "SELECT DISTINCT State FROM StateNames ORDER BY State;"
df_state_codes = pd.read_sql_query(query, conn)
print(f"Number of distinct state codes: {df_state_codes['State'].nunique()}")
display(styled_table(df_state_codes, "Distinct State Codes", "#2E7D32"))


Number of distinct state codes: 51

,State
0,AK
1,AL
2,AR
3,AZ
4,CA
5,CO
6,CT
7,DC
8,DE
9,FL


## Cleaning Summary

All validation checks passed successfully for **both** `NationalNames` and `StateNames`.

The dataset contains:

- no missing values
- no duplicate records
- valid year values
- valid gender values
- valid birth counts
- no empty names
- expected number of distinct state codes

No cleaning operations were required.


In [26]:
processed = sqlite3.connect("../data/cleaned/names_clean.sqlite")

df_national.to_sql("NationalNames", processed, index=False, if_exists="replace")
df_state.to_sql("StateNames", processed, index=False, if_exists="replace")

processed.close()
conn.close()
